# TP — Benchmark CPU ResNet-18 / CIFAR-10

**Protocole commun :** modèle ResNet-18 fine-tuné à 10 classes, 50 images de test CIFAR-10, mêmes indices, prétraitement issu de `ResNet18_Weights.DEFAULT.transforms()`, batch size 1, FP32, 5 répétitions, warm-up exclu des mesures.

**Prérequis :** avoir exécuté la phase 1 et sauvegardé `resnet18_cifar10.pth`, `benchmark_indices.json` et éventuellement `reference_logits.npy` dans `MyDrive/AI_Deployment_TP/shared`. Exécuter avec **Runtime → Change runtime type → CPU**. Les fichiers produits sont copiés sur Drive.

In [ ]:
import os, json, time, random, platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torchvision
from torchvision import datasets, models
from torchvision.models import ResNet18_Weights
from torch.utils.data import DataLoader, Subset
from google.colab import drive

drive.mount('/content/drive')
SHARED = Path('/content/drive/MyDrive/AI_Deployment_TP/shared')
assert (SHARED/'resnet18_cifar10.pth').exists(), 'Exécutez la phase 1 et sauvegardez le modèle dans Drive.'
assert (SHARED/'benchmark_indices.json').exists(), 'Fichier benchmark_indices.json manquant.'
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(min(4, os.cpu_count() or 1))
DEVICE=torch.device('cpu')
print('CPU:', platform.processor(), '| threads:', torch.get_num_threads())
print('torch:',torch.__version__, '| torchvision:',torchvision.__version__)

In [ ]:
weights=ResNet18_Weights.DEFAULT
preprocess=weights.transforms()
test_dataset=datasets.CIFAR10(root='/content/data',train=False,download=True,transform=preprocess)
indices=json.loads((SHARED/'benchmark_indices.json').read_text())
assert len(indices)==50 and len(set(indices))==50
loader=DataLoader(Subset(test_dataset,indices),batch_size=1,shuffle=False,num_workers=0)
images=torch.cat([x for x,_ in loader],dim=0) # [50,3,H,W]
labels=np.array([int(y.item()) for _,y in loader])
print('Images:',tuple(images.shape),'Labels:',labels.shape)

In [ ]:
model=models.resnet18(weights=None)
model.fc=torch.nn.Linear(model.fc.in_features,10)
model.load_state_dict(torch.load(SHARED/'resnet18_cifar10.pth',map_location='cpu',weights_only=True))
model.eval()
with torch.inference_mode():
    reference_logits=torch.cat([model(x) for x in images.split(1)],dim=0).numpy()
reference_pred=reference_logits.argmax(axis=1)
print('Référence PyTorch Top-1:',round(100*np.mean(reference_pred==labels),2),'%')
if (SHARED/'reference_logits.npy').exists():
    saved=np.load(SHARED/'reference_logits.npy')
    print('Écart max vs référence phase 1:',float(np.max(np.abs(saved-reference_logits))))

## Protocole de mesure
Chaque répétition mesure **l'inférence des 50 images**, après warm-up. La latence par image est la durée totale divisée par 50 (elle comprend l'appel Python/runtime). Les compilations et conversions sont exclues. Les répétitions sont réalisées dans la même session Colab ; noter que cela ne mesure pas la variabilité entre machines ou sessions.

In [ ]:
def evaluate_and_benchmark(predict_one, name, repetitions=5, warmup=10):
    # predict_one : prend un numpy float32 [1,3,H,W] et retourne numpy [1,10]
    batches=[images[i:i+1].numpy().astype(np.float32,copy=False) for i in range(len(images))]
    for _ in range(warmup): predict_one(batches[0])
    rows=[]; logits_runs=[]
    for run in range(repetitions):
        outputs=[]
        start=time.perf_counter()
        for batch in batches:
            outputs.append(np.asarray(predict_one(batch)))
        elapsed=time.perf_counter()-start
        logits=np.concatenate(outputs,axis=0)
        logits_runs.append(logits)
        preds=logits.argmax(axis=1)
        rows.append(dict(pipeline=name,run=run+1,total_ms=elapsed*1000,
                         latency_ms_per_image=elapsed*1000/len(batches),
                         throughput_images_s=len(batches)/elapsed,
                         top1_accuracy_pct=100*np.mean(preds==labels),
                         agreement_with_pytorch_pct=100*np.mean(preds==reference_pred),
                         max_abs_logit_diff=float(np.max(np.abs(logits-reference_logits)))))
    df=pd.DataFrame(rows)
    stats=df.select_dtypes(include='number').drop(columns='run').agg(['mean','std','min','max'])
    return df,stats,logits_runs[0]

def save_results(df,stats,logits,stem):
    dest=Path('/content/drive/MyDrive/AI_Deployment_TP/results'); dest.mkdir(parents=True,exist_ok=True)
    df.to_csv(dest/f'{stem}.csv',index=False)
    stats.to_csv(dest/f'{stem}_stats.csv')
    np.save(dest/f'{stem}_logits.npy',logits)
    fig,axs=plt.subplots(1,3,figsize=(15,4))
    for ax,(col,title) in zip(axs,[('throughput_images_s','Débit (images/s)'),('latency_ms_per_image','Latence (ms/image)'),('top1_accuracy_pct','Top-1 (%)')]):
        ax.bar([stem],[df[col].mean()],yerr=[df[col].std()],capsize=5)
        ax.set_title(title); ax.grid(axis='y',alpha=.3)
    plt.tight_layout(); plt.savefig(dest/f'{stem}_plots.png',dpi=160); plt.show()
    print('Résultats sauvegardés :',dest)


## Pipeline CPU 2 — PyTorch → torch.compile (Inductor) → CPU
`torch.compile` utilise le compilateur natif PyTorch. **Attention :** la première compilation peut prendre plusieurs minutes et nécessiter une version Colab compatible de PyTorch/Triton. Le temps de compilation est exclu du benchmark.

In [ ]:
import torch
print('torch.compile disponible:',hasattr(torch,'compile'))
start_compile=time.perf_counter()
compiled_model=torch.compile(model,backend='inductor',mode='default')
try:
    with torch.inference_mode():
        check=compiled_model(images[:1]).numpy()
except Exception as exc:
    raise RuntimeError('Échec de torch.compile/Inductor sur ce runtime CPU. Vérifier la version PyTorch et les logs ; ne pas remplacer silencieusement par PyTorch eager.') from exc
compile_first_call_s=time.perf_counter()-start_compile
print('Compilation + premier appel (s):',round(compile_first_call_s,2))
print('Différence max vs PyTorch:',float(np.max(np.abs(check-reference_logits[:1]))))

In [ ]:
def predict_compiled(x):
    with torch.inference_mode():
        return compiled_model(torch.from_numpy(x)).numpy()

df,stats,logits=evaluate_and_benchmark(predict_compiled,'CPU_TorchCompile_Inductor')
display(df); display(stats)
save_results(df,stats,logits,'cpu_torchcompile')

### Analyse à compléter
Commentez le coût initial de compilation séparément de la latence d'inférence, la variabilité des répétitions, la concordance des logits et la précision Top-1. **Énergie : non disponible** sans instrumentation fiable.